# AutoMorphClass on three shapes, biomarker by biomarker

A close reading of one implementation. The
[main notebook](biomarker-synthetic.ipynb) compares six programs across nine shapes and answers
*which of these agree*; this one asks a narrower question about
[AutoMorphClass](../docs/projects/automorphclass.md) alone — **what does it actually return, on the three
simplest shapes, at each angle, beside what the geometry requires?**

Everything is under **AutoMorphClass's own column names**, not the catalogued ones. That is the point:
the catalogued names exist to make two programs comparable, and there is only one program here, so
the names its authors chose are the ones a reader checking against their documentation will look
for. Where a column maps onto a catalogued biomarker the mapping is shown beside it, and where the
shape settles a theoretical value that is shown too.

**The three shapes, in order of how much they ask for:**

| Shape | What it is | Why it comes first |
| --- | --- | --- |
| `straight` | one straight vessel per class, leaving the optic disc | every curvature measure must read zero and every tortuosity exactly 1 |
| `arc` | a circular arc per class | curvature is exactly 1/r, and the arc-to-chord ratio has a closed form |
| `sinusoid` | a sine wave per class | the only one of the three whose curvature changes sign |

A blank in the theory column means **this shape settles no value for that quantity** — which is not
the same as the quantity being wrong, and not the same as AutoMorph declining to answer it.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def repository() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))

from benchmarks import biomarker_synthetic as benchmark  # noqa: E402
from benchmarks.shapes import store  # noqa: E402
from biomarkers.utils import catalogue  # noqa: E402

#: The one implementation this notebook reads. Change it and everything below follows.
IMPLEMENTATION = "automorphclass"

#: In order of how much they ask for: a straight vessel, then a curve, then a curve that changes
#: its mind about which way it bends.
SHAPES = ("straight", "arc", "sinusoid")

EVIDENCE = ROOT / "results" / benchmark.NAME / IMPLEMENTATION
STORE = ROOT / store.STORE
pd.set_option("display.width", 200, "display.max_rows", 200)

#: AutoMorph's own column name → the catalogued biomarker it answers to, or None where the
#: catalogue has no name for it. This is the adapter's claim, not the run's.
NAMES = catalogue.load(IMPLEMENTATION).declare()["names"]

TRUTH = pd.read_csv(STORE / "ground_truth.csv").set_index("key")
print(f"{IMPLEMENTATION}: {len(NAMES)} columns, "
      f"{sum(1 for v in NAMES.values() if v)} of them with a catalogued name")

automorphclass: 18 columns, 15 of them with a catalogued name


In [2]:
def readings(shape: str) -> pd.DataFrame:
    """What AutoMorph returned for one shape: a row per column, a column per angle.

    The theoretical value is joined on last and only where the shape settles one, so a blank there
    is a statement about the fixture rather than about AutoMorph.
    """
    measured = pd.read_csv(EVIDENCE / f"{shape}.csv")
    rows = {}
    for own in NAMES:
        values = {}
        for _, rendering in measured.iterrows():
            values[f"{rendering['rotation']:.0f}°"] = rendering.get(f"said_{own}")
        canonical = NAMES[own]
        truth = np.nan
        if canonical:
            keys = [k for k in TRUTH.index if k.startswith(f"{shape}-")]
            if keys and canonical in TRUTH.columns:
                truth = TRUTH.loc[keys[0], canonical]
        rows[own] = {
            **values,
            "theory": truth,
            "catalogued as": canonical or "—",
        }
    frame = pd.DataFrame(rows).T
    frame.index.name = f"{IMPLEMENTATION}'s own name"
    return frame


def show(shape: str) -> pd.DataFrame:
    frame = readings(shape)
    angles = [c for c in frame.columns if c.endswith("°")]
    numeric = frame[angles + ["theory"]].astype(float)
    spread = numeric[angles].max(axis=1) - numeric[angles].min(axis=1)
    out = numeric.round(4)
    out["moves"] = spread.round(4)
    out["catalogued as"] = frame["catalogued as"]
    return out

## 1. `straight`

A single straight vessel per class, leaving the optic disc. **Everything about it is known
exactly**: the arc-to-chord ratio is 1, every curvature measure is 0, and the width is the width it
was drawn at. Nothing here is a hard measurement, which is what makes it the right place to start —
a value that is wrong on a straight line is wrong for a reason that has nothing to do with
difficulty.

In [3]:
show("straight")

,0°,30°,60°,90°,theory,moves,catalogued as
automorphclass's own name,,,,,,,
vessel_density_vessels,0.0060,0.0060,0.0060,0.0060,0.006,0.0000,vascular-density/over-image/vessels
fractal_dimension_vessels,0.9005,1.0008,1.0124,0.9005,NaN,0.1119,fractal-dimension/box-counting/vessels
average_width_vessels,20.5583,23.6689,23.6439,20.5918,NaN,3.1106,vessel-calibre/mean-width/vessels
distance_tortuosity_vessels,1.0007,1.0739,1.0731,1.0007,NaN,0.0733,tortuosity/hart-tau1/vessels
squared_curvature_tortuosity_vessels,0.2527,5.9254,5.4813,0.2068,NaN,5.7186,—
tortuosity_density_vessels,0.0001,0.0102,0.0073,0.0000,NaN,0.0102,tortuosity/grisan-density/vessels
vessel_density_artery,0.0024,0.0024,0.0024,0.0024,NaN,0.0000,vascular-density/over-image/artery
fractal_dimension_artery,0.3758,0.8849,0.8849,0.3758,NaN,0.5091,fractal-dimension/box-counting/artery
average_width_artery,16.2825,18.8199,18.8553,16.3355,16.000,2.5728,vessel-calibre/mean-width/artery


## 2. `arc`

A circular arc per class, the two sharing a subtended angle and differing in radius. The
arc-to-chord ratio is `θ / (2 sin(θ/2))` in closed form and **independent of radius**, so both
classes must return the same tortuosity while their curvatures differ. That pairing is the test:
an implementation that returns different tortuosities for the two is measuring the radius somewhere
it should not.

In [4]:
show("arc")

,0°,30°,60°,90°,theory,moves,catalogued as
automorphclass's own name,,,,,,,
vessel_density_vessels,0.0059,0.0059,0.0059,0.0059,0.0059,0.0000,vascular-density/over-image/vessels
fractal_dimension_vessels,1.0194,1.0255,1.0713,1.0194,NaN,0.0519,fractal-dimension/box-counting/vessels
average_width_vessels,22.2367,22.2167,22.1478,22.2367,NaN,0.0889,vessel-calibre/mean-width/vessels
distance_tortuosity_vessels,1.1700,1.1711,1.1718,1.1724,NaN,0.0025,tortuosity/hart-tau1/vessels
squared_curvature_tortuosity_vessels,10.5420,10.4763,10.5204,10.7824,NaN,0.3060,—
tortuosity_density_vessels,0.0089,0.0091,0.0087,0.0091,NaN,0.0004,tortuosity/grisan-density/vessels
vessel_density_artery,0.0027,0.0027,0.0027,0.0027,NaN,0.0000,vascular-density/over-image/artery
fractal_dimension_artery,0.9345,0.9369,0.9803,0.9345,NaN,0.0458,fractal-dimension/box-counting/artery
average_width_artery,18.1656,18.0549,18.0580,18.1370,16.0000,0.1108,vessel-calibre/mean-width/artery


## 3. `sinusoid`

A sine wave per class, and **the only one of the three whose curvature changes sign**. Its arc
length is an elliptic integral with no elementary closed form, so the theoretical value here is the
integral rather than a formula. It is the shape that settles the inflection count and Grisan's
density, both of which are zero on everything else.

In [5]:
show("sinusoid")

,0°,30°,60°,90°,theory,moves,catalogued as
automorphclass's own name,,,,,,,
vessel_density_vessels,0.0066,0.0066,0.0066,0.0066,0.0066,0.0000,vascular-density/over-image/vessels
fractal_dimension_vessels,1.0281,1.0087,1.0467,1.0281,NaN,0.0380,fractal-dimension/box-counting/vessels
average_width_vessels,24.7138,21.5448,22.2617,24.7581,NaN,3.2132,vessel-calibre/mean-width/vessels
distance_tortuosity_vessels,1.4352,1.4441,1.4601,1.4336,NaN,0.0265,tortuosity/hart-tau1/vessels
squared_curvature_tortuosity_vessels,8.4608,13.9437,12.7357,7.9740,NaN,5.9697,—
tortuosity_density_vessels,0.0084,0.0099,0.0094,0.0083,NaN,0.0016,tortuosity/grisan-density/vessels
vessel_density_artery,0.0026,0.0026,0.0026,0.0026,NaN,0.0000,vascular-density/over-image/artery
fractal_dimension_artery,0.9013,0.8998,0.9325,0.9013,NaN,0.0327,fractal-dimension/box-counting/artery
average_width_artery,20.0128,17.3181,17.3391,20.0128,16.0000,2.6946,vessel-calibre/mean-width/artery


## 4. What the three tables show

AutoMorphClass is a rewrite of [AutoMorph](biomarker-synthetic-automorph.ipynb)'s measuring stage,
and the most useful way to read these tables is against that one.

### 4.1 Every reading is a possible value, which is not true of its ancestor

AutoMorph returns **exactly 0.0000** for `distance_tortuosity` on `straight` at 30° and 60°, and on
`sinusoid` at 0° and 90°. An arc over a chord cannot be less than 1, so those are failures reported
as numbers. **Nothing in these three tables is impossible**: every tortuosity is at or above 1,
every width is positive, every fractal dimension is between 0 and 2. Whatever else follows, the
rewrite fixed the thing that most needed fixing, because a wrong number you can see is worth more
than a failure you cannot.

### 4.2 Steady, and biased high

On `arc`, where the geometry requires **1.1107** at every angle and for both classes:

| | 0° | 30° | 60° | 90° | theory | moves |
| --- | --- | --- | --- | --- | --- | --- |
| `distance_tortuosity_artery` | 1.1697 | 1.1704 | 1.1715 | 1.1739 | 1.1107 | 0.0042 |
| `distance_tortuosity_vein` | 1.1703 | 1.1722 | 1.1723 | 1.1704 | 1.1107 | 0.0020 |

Turning the picture moves it by **0.4%** — and it is **5.3% too high at every angle**. The two
classes agree with each other to three decimals, which is what the geometry requires of them, and
both agree on the wrong value. That is the opposite failure from AutoMorph, whose vein reads 1.037,
1.062, 7.133 and 5.367 on the same shape: a steady bias can be corrected once it is known, and
noise of that size cannot be corrected at all.

The same holds on `sinusoid` — 1.43 to 1.46 against a required 1.3726, about 5% high and moving by
2% — and on `straight`, where it reads 1.0007 at the axis-aligned angles and 1.073 at the diagonal
ones.

### 4.3 The width error runs the opposite way to AutoMorph's

On `straight`, one unchanging vessel at four angles:

| | 0° | 30° | 60° | 90° | drawn at |
| --- | --- | --- | --- | --- | --- |
| `average_width_vein`, **AutoMorphClass** | 24.87 | 28.52 | 28.41 | 24.87 | 24 |
| `average_width_vein`, **AutoMorph** | 14.15 | 24.94 | 25.13 | 14.15 | 24 |

Both wobble with the angle and they wobble **in opposite directions**. AutoMorphClass is nearly
exact where the vessel is axis-aligned and 18% wide where it is diagonal; AutoMorph is nearly exact
where it is diagonal and 41% narrow where it is axis-aligned. One of the two rewrote the width
calculation and the tables cannot say which is nearer right in general — only that on this shape
the rewrite is closer, and that the two cannot both be measuring what they claim to measure.

### 4.4 A straight line with a fractal dimension of 0.376

`fractal_dimension_artery` on `straight` reads **0.3758, 0.8849, 0.8849, 0.3758**. A straight line
has dimension 1, and a bounded raster of one will read a little under it — 0.88 is a plausible
underestimate and 0.376 is not a dimension of anything.

**AutoMorph does not do this.** Its `fractal_dimension_artery` on the same shape reads 0.927,
0.977, 0.977, 0.927. So this is something the rewrite introduced rather than something it
inherited, and it appears only at the two axis-aligned angles.

### 4.5 What is not checked here

Every fractal dimension, both per-class vessel densities and `squared_curvature_tortuosity` carry
no theory on these shapes — the first two because these shapes settle no value for them, the last
because the catalogue has no name for it at all. Those rows are reported and unchecked, which is a
third state and not a pass.
